## Data Profiling

Dataset: SIAGA Pantura (flood_dataset.parquet), sumber Hugging Face ethanchrstian/siaga-pantura-hazard-data.

In [1]:
import polars as pl
import duckdb

DATA_PATH = "../data/raw/flood_dataset.parquet"

DATA_PATH

'../data/raw/flood_dataset.parquet'

### 1. Jumlah baris, kecamatan, periode

In [2]:
df = pl.scan_parquet(DATA_PATH)

df.select(
    pl.len().alias("baris"),
    pl.col("district_id").n_unique().alias("kecamatan"),
    pl.col("date").min().alias("awal"),
    pl.col("date").max().alias("akhir"),
).collect()

baris,kecamatan,awal,akhir
u32,u32,datetime[μs],datetime[μs]
1152432,318,2015-01-30 00:00:00,2024-12-31 00:00:00


### 2. Skema dan tipe data

In [3]:
schema = df.collect_schema()
pl.DataFrame({"kolom": schema.names(), "tipe": [str(t) for t in schema.dtypes()]})

kolom,tipe
str,str
"""district_id""","""String"""
"""date""","""Datetime(time_unit='us', time_…"
"""year""","""Int32"""
"""rain_1d""","""Float64"""
"""rain_3d""","""Float64"""
…,…
"""disc_7d_mean""","""Float64"""
"""disc_change_3d""","""Float64"""
"""month""","""Int32"""


In [4]:
df.head(5).collect()

district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
str,datetime[μs],i32,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64
"""IDN.10.13.10_1""",2015-01-30 00:00:00,2015,13.3,35.0,62.0,348.3,25.98,26.238571,0.56,1,1,0.0
"""IDN.10.13.10_1""",2015-01-31 00:00:00,2015,12.5,44.0,67.2,348.1,26.35,25.92,2.35,1,1,0.0
"""IDN.10.13.10_1""",2015-02-01 00:00:00,2015,10.4,36.2,67.0,352.5,25.98,25.548571,1.8,2,1,0.0
"""IDN.10.13.10_1""",2015-02-02 00:00:00,2015,1.9,24.8,65.9,326.8,24.76,25.238571,-1.22,2,1,0.0
"""IDN.10.13.10_1""",2015-02-03 00:00:00,2015,20.7,33.0,80.5,327.8,25.98,25.318571,-0.37,2,1,0.0


### 3. Missing value per kolom

In [5]:
df.null_count().collect().transpose(include_header=True, column_names=["jumlah_null"])

column,jumlah_null
str,u32
"""district_id""",0
"""date""",0
"""year""",0
"""rain_1d""",0
"""rain_3d""",0
…,…
"""disc_7d_mean""",0
"""disc_change_3d""",0
"""month""",0


### 4. Duplikat

In [6]:
total = df.select(pl.len()).collect().item()
unik = df.unique(subset=["district_id", "date"]).select(pl.len()).collect().item()

print(f"baris={total:,} pasangan unik (district_id, date)={unik:,} duplikat={total - unik:,}")

baris=1,152,432 pasangan unik (district_id, date)=1,152,432 duplikat=0


### 5. Ringkasan statistik

In [7]:
summary = duckdb.sql(f"SUMMARIZE SELECT * FROM read_parquet('{DATA_PATH}')").pl()
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""district_id""","""VARCHAR""","""IDN.10.13.10_1""","""IDN.9.9.9_1""",347,null,null,null,null,null,1152432,0.00
"""date""","""TIMESTAMP""","""2015-01-30 00:00:00""","""2024-12-31 00:00:00""",4543,"""2020-01-15 12:00:00""",null,"""2017-07-23 03:35:52.340014""","""2020-01-15 22:21:26.672156""","""2022-07-09 02:44:46.356104""",1152432,0.00
"""year""","""INTEGER""","""2015""","""2024""",11,"""2019.5364238410596""","""2.8556717940747394""","""2017""","""2020""","""2022""",1152432,0.00
"""rain_1d""","""DOUBLE""","""0.0""","""258.7""",1546,"""6.793897687671181""","""10.10181064199661""","""0.3102058265651896""","""2.97825380952829""","""9.777347223340776""",1152432,0.00
"""rain_3d""","""DOUBLE""","""0.0""","""415.70000000000005""",56631,"""20.382071566912654""","""23.51372439659673""","""2.201293751937434""","""13.672714821062801""","""30.88169041821603""",1152432,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""disc_7d_mean""","""DOUBLE""","""0.0""","""1307.3185714285714""",179705,"""20.806564402186886""","""70.15783131833687""","""0.7626605869124681""","""2.7895883919333695""","""10.820594251990162""",1152432,0.00
"""disc_change_3d""","""DOUBLE""","""-758.7300000000001""","""1076.38""",76977,"""0.0009839279020340383""","""18.861786504809054""","""-0.6191748743394475""","""-0.03021803170823922""","""0.3096824373378738""",1152432,0.00
"""month""","""INTEGER""","""1""","""12""",13,"""6.566501103752759""","""3.4269421011485166""","""4""","""7""","""10""",1152432,0.00
